### Imports

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import joblib
import pandas as pd

from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder

from src.features import FeatureEngineer

### Préparation de la donnée

In [3]:
DATA_DIR = PROJECT_ROOT / "data"

extrait_eval = pd.read_csv(DATA_DIR / "extrait_eval.csv")
extrait_sirh = pd.read_csv(DATA_DIR / "extrait_sirh.csv")
extrait_sondage = pd.read_csv(DATA_DIR / "extrait_sondage.csv")

In [4]:
def eval_number_integer(valeur):
    valeur=valeur.replace("E_","")
    valeur=int(valeur)
    return(valeur)

In [5]:
extrait_eval['eval_number'] = extrait_eval['eval_number'].apply(eval_number_integer)

In [6]:
def pourcentage_vers_int(valeur):
    valeur = valeur.replace("%", "")
    valeur = valeur.strip()
    valeur = int(valeur)
    return valeur

In [7]:
extrait_eval = extrait_eval.rename(columns={
    'augementation_salaire_precedente': 'augmentation_salaire_precedente'
})
extrait_eval['augmentation_salaire_precedente'] = extrait_eval['augmentation_salaire_precedente'].apply(pourcentage_vers_int)

In [8]:
eval_quantitatives = [
    "augmentation_salaire_precedente"
]

eval_qualitatives_ordinales = [
    "satisfaction_employee_environnement",
    "note_evaluation_precedente",
    "niveau_hierarchique_poste",
    "satisfaction_employee_nature_travail",
    "satisfaction_employee_equipe",
    "satisfaction_employee_equilibre_pro_perso",
    "note_evaluation_actuelle"
]

eval_qualitatives_nominales = [
    "heure_supplementaires"
]

eval_identifiants = [
    "eval_number"
]

In [9]:
extrait_sirh = extrait_sirh.rename(columns={
    'id_employee': 'id_employe'
})

In [10]:
extrait_sirh = extrait_sirh.drop(columns='nombre_heures_travailless')

In [11]:
sirh_quantitatives = [
    "age",
    "revenu_mensuel",
    "nombre_experiences_precedentes",
    "annee_experience_totale",
    "annees_dans_l_entreprise",
    "annees_dans_le_poste_actuel"
]

sirh_qualitatives = [
    "genre",
    "statut_marital",
    "departement",
    "poste"
]

sirh_identifiants = [
    "id_employe"
]

In [12]:
extrait_sondage = extrait_sondage.drop(columns=['nombre_employee_sous_responsabilite', 'ayant_enfants'])
print(f"Après suppression des colonnes, il y a {extrait_sondage.shape[1]} colonnes dans le dataset")

Après suppression des colonnes, il y a 10 colonnes dans le dataset


In [13]:
extrait_sondage = extrait_sondage.rename(columns={
    "annes_sous_responsable_actuel": "annees_sous_responsable_actuel"
})

In [14]:
target = "a_quitte_l_entreprise"

sondage_quantitatives = [
    "nombre_participation_pee",
    "nb_formations_suivies",
    "distance_domicile_travail",
    "annees_depuis_la_derniere_promotion",
    "annees_sous_responsable_actuel"
]

sondage_qualitatives_nominales = [
    "domaine_etude"
]

sondage_qualitatives_ordinales = [
    "niveau_education",
    "frequence_deplacement"
]

sondage_identifiants = [
    "code_sondage",
]

In [15]:
extrait = (
    extrait_eval
    .merge(extrait_sirh, left_on='eval_number', right_on='id_employe')
    .merge(extrait_sondage, left_on='id_employe', right_on='code_sondage')
)
extrait.head()

,satisfaction_employee_environnement,note_evaluation_precedente,niveau_hierarchique_poste,satisfaction_employee_nature_travail,satisfaction_employee_equipe,satisfaction_employee_equilibre_pro_perso,eval_number,note_evaluation_actuelle,heure_supplementaires,augmentation_salaire_precedente,...,a_quitte_l_entreprise,nombre_participation_pee,nb_formations_suivies,code_sondage,distance_domicile_travail,niveau_education,domaine_etude,frequence_deplacement,annees_depuis_la_derniere_promotion,annees_sous_responsable_actuel
0,2,3,2,4,1,1,1,3,Oui,11,...,Oui,0,0,1,1,2,Infra & Cloud,Occasionnel,0,5
1,3,2,2,2,4,3,2,4,Non,23,...,Non,1,3,2,8,1,Infra & Cloud,Frequent,1,7
2,4,2,1,3,2,3,4,3,Oui,15,...,Oui,0,3,4,2,2,Autre,Occasionnel,0,0
3,4,3,1,3,3,3,5,3,Oui,11,...,Non,0,3,5,3,4,Infra & Cloud,Frequent,3,0
4,1,3,1,2,4,3,7,3,Non,12,...,Non,1,3,7,2,1,Transformation Digitale,Occasionnel,2,2


In [16]:
extrait = extrait.drop(columns=['eval_number', 'code_sondage'])
print(f"Nous avons {extrait.shape[0]} lignes et {extrait.shape[1]} colonnes dans notre dataset")

Nous avons 1470 lignes et 29 colonnes dans notre dataset


In [17]:
col_quantitatives = eval_quantitatives + sirh_quantitatives + sondage_quantitatives
col_qualitatives_ordinales = eval_qualitatives_ordinales + sondage_qualitatives_ordinales
col_qualitatives_nominales = eval_qualitatives_nominales + sondage_qualitatives_nominales + sirh_qualitatives

In [18]:
extrait['a_quitte_l_entreprise'] = extrait['a_quitte_l_entreprise'].map({
    'Oui': 1, 'Non': 0
})

In [19]:
extrait = extrait.drop(columns=['niveau_hierarchique_poste', 'annee_experience_totale',
                                'annees_dans_le_poste_actuel', 'annees_sous_responsable_actuel',
                                'augmentation_salaire_precedente'])

In [20]:
colonnes_a_supprimer = [
    'annee_experience_totale',
    'annees_dans_le_poste_actuel',
    'annees_sous_responsable_actuel',
    'augmentation_salaire_precedente'
]

col_quantitatives = [
    col for col in col_quantitatives
    if col not in colonnes_a_supprimer
]

In [21]:
col_qualitatives_ordinales = [col for col in col_qualitatives_ordinales if col != 'niveau_hierarchique_poste']

In [22]:
extrait = extrait.drop(columns=['departement'])
col_qualitatives_nominales = [col for col in col_qualitatives_nominales if col != 'departement']

In [23]:
extrait.to_csv(DATA_DIR / "dataset_final.csv", index=False)

In [24]:
y = extrait['a_quitte_l_entreprise']
X = extrait.drop(columns=['a_quitte_l_entreprise', 'id_employe'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cols_ordinales_a_encoder = ["frequence_deplacement"]

cols_numeriques = [col for col in X.columns 
                   if col not in col_qualitatives_nominales
                   and col not in cols_ordinales_a_encoder]

preprocessor = ColumnTransformer([
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'), 
     col_qualitatives_nominales),
    ('ordinal', OrdinalEncoder(categories=[["Aucun", "Occasionnel", "Frequent"]]), 
     cols_ordinales_a_encoder),
    ('num', Pipeline([
        ('scaler', StandardScaler())
        ]), cols_numeriques),
])

### Entrainement du modèle LightGBM optimisé

In [25]:
meilleurs_params = {
    "n_estimators": 165,
    "learning_rate": 0.1103,
    "num_leaves": 85,
    "max_depth": 11,
    "min_child_samples": 65,
    "reg_alpha": 0.0000,
    "reg_lambda": 2.2501,
    "class_weight": "balanced",
    "random_state": 42,
    "verbosity": -1,
}

In [26]:
pipeline_optimise = Pipeline([
    ("feature_engineer", FeatureEngineer()),
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier(**meilleurs_params)),
])

In [27]:
pipeline_optimise.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('feature_engineer', ...), ('preprocessor', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
Name,Type,Value
mediane_par_education_,"Series[float64](5,)",niveau_educat...dtype: float64
mediane_par_poste_,"Series[float64](9,)",poste Assista...dtype: float64
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('onehot', ...), ('ordinal', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By settin

### Export du modèle

In [28]:
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(
    {
        "pipeline": pipeline_optimise,
        "seuil": 0.371,
        "colonnes_entree": list(X_train.columns),
    },
    MODEL_DIR / "modele_lightgbm_attrition.joblib",
)

['/Users/matthieu/dev/openclassrooms/projet5/models/modele_lightgbm_attrition.joblib']